# DCOPF Example 4A - PTDF-Based Formulation (Data from File)
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (e4A_M2 vs e4):** Same PTDF-based formulation. Data is loaded from an external file instead of hard-coded. Uses system-wide power balance (`sysPowerBalance`) instead of nodal balance.

In [ ]:
from pyomo.environ import *
import re
import time

# ─────────────────────────────────────────────────────────────────
# Base path & data file
# ─────────────────────────────────────────────────────────────────
BASE_PATH = '/Users/csab/Desktop/ECE6379_Python_Code/20_DCOPF/'
data_file = BASE_PATH + 'DCOPF_IC_e4-e4A_M2_data.txt'

BaseMW = 100
refBus = 3

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Parser for AMPL-style data files
# Supports:
#   1D param block : param: SET_NAME: col1 col2 ... :=
#   2D param block : param NAME: col1 col2 ... :=  (matrix / PTDF form)
#   Inline comments: # ...
# ─────────────────────────────────────────────────────────────────
def parse_ampl_data(filepath):
    with open(filepath, 'r') as f:
        raw_lines = f.readlines()

    clean_lines = [re.sub(r'#.*', '', l).strip() for l in raw_lines]
    clean_lines = [l for l in clean_lines if l]
    text = ' '.join(clean_lines)

    blocks = [b.strip() for b in re.split(r';', text) if b.strip()]

    sets   = {}
    params = {}

    for block in blocks:
        # ── 1D block: param: SET_NAME: col1 col2 ... := row_data ──
        m = re.match(r'param\s*:\s*(\w+)\s*:\s*([\w\s]+):=\s*(.*)', block, re.S)
        if m:
            set_name  = m.group(1).strip()
            col_names = m.group(2).split()
            tokens    = m.group(3).split()
            idx_list  = []
            for col in col_names:
                params.setdefault(col, {})
            it = iter(tokens)
            try:
                while True:
                    idx = int(next(it))
                    idx_list.append(idx)
                    for col in col_names:
                        params[col][idx] = float(next(it))
            except StopIteration:
                pass
            sets[set_name] = idx_list
            continue

        # ── 2D block: param NAME: col1 col2 ... := row_data ───────
        m = re.match(r'param\s+(\w+)\s*:\s*([\d\s]+):=\s*(.*)', block, re.S)
        if m:
            param_name  = m.group(1).strip()
            col_indices = [int(c) for c in m.group(2).split()]
            tokens      = m.group(3).split()
            param_dict  = {}
            it = iter(tokens)
            try:
                while True:
                    row_idx = int(next(it))
                    for col_idx in col_indices:
                        param_dict[(row_idx, col_idx)] = float(next(it))
            except StopIteration:
                pass
            params[param_name] = param_dict
            continue

    return sets, params


# ─────────────────────────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────────────────────────
sets, params = parse_ampl_data(data_file)

BUS_data    = sets['BUS']
GEN_data    = sets['GEN']
BRANCH_data = sets['BRANCH']

busLoad_data       = {int(k): v      for k, v in params['busLoad'].items()}
gen_bus_data       = {int(k): int(v) for k, v in params['gen_bus'].items()}
gen_min_data       = {int(k): v      for k, v in params['gen_min'].items()}
gen_max_data       = {int(k): v      for k, v in params['gen_max'].items()}
gen_OpCost_data    = {int(k): v      for k, v in params['gen_OpCost'].items()}
branch_frmbus_data = {int(k): int(v) for k, v in params['branch_frmbus'].items()}
branch_tobus_data  = {int(k): int(v) for k, v in params['branch_tobus'].items()}
branch_x_data      = {int(k): v      for k, v in params['branch_x'].items()}
branch_rate_data   = {int(k): v      for k, v in params['branch_rate'].items()}
PTDF_data          = {(int(k[0]), int(k[1])): v for k, v in params['PTDF'].items()}

print('BUS    :', BUS_data)
print('GEN    :', GEN_data)
print('BRANCH :', BRANCH_data)
print('PTDF   :', PTDF_data)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.BUS    = Set(initialize=BUS_data)
model.BRANCH = Set(initialize=BRANCH_data)
model.GEN    = Set(initialize=GEN_data)

# Parameters
model.busLoad       = Param(model.BUS,    initialize=busLoad_data)
model.gen_bus       = Param(model.GEN,    initialize=gen_bus_data,        within=model.BUS)
model.gen_min       = Param(model.GEN,    initialize=gen_min_data)
model.gen_max       = Param(model.GEN,    initialize=gen_max_data)
model.gen_OpCost    = Param(model.GEN,    initialize=gen_OpCost_data)
model.branch_frmbus = Param(model.BRANCH, initialize=branch_frmbus_data,  within=model.BUS)
model.branch_tobus  = Param(model.BRANCH, initialize=branch_tobus_data,   within=model.BUS)
model.branch_x      = Param(model.BRANCH, initialize=branch_x_data)
model.branch_rate   = Param(model.BRANCH, initialize=branch_rate_data)
model.PTDF          = Param(model.BRANCH, model.BUS, initialize=PTDF_data)

# Decision Variables
model.G  = Var(model.GEN)
model.pk = Var(model.BRANCH)

# ── Objective ────────────────────────────────────────────────────
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
# Branch thermal limits
def branchLimits_rule(model, k):
    return inequality(-model.branch_rate[k], model.pk[k], model.branch_rate[k])
model.branchLimits = Constraint(model.BRANCH, rule=branchLimits_rule)

# PTDF-based line flow: pk = sum_n PTDF[k,n] * (Pg_n - Pd_n)
def lineFlow_rule(model, k):
    return model.pk[k] == sum(
        model.PTDF[k, n] * (
            sum(model.G[g] for g in model.GEN if model.gen_bus[g] == n)
            - model.busLoad[n]
        )
        for n in model.BUS
    )
model.lineFlow = Constraint(model.BRANCH, rule=lineFlow_rule)

# Generator capacity limits
def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

# System-wide power balance
def sysPowerBalance_rule(model):
    return sum(model.G[g] for g in model.GEN) == sum(model.busLoad[n] for n in model.BUS)
model.sysPowerBalance = Constraint(rule=sysPowerBalance_rule)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\nG:')
for g in model.GEN:
    print(f'  G[{g}] = {model.G[g].value}')

print('\npk:')
for k in model.BRANCH:
    print(f'  pk[{k}] = {model.pk[k].value}')

print(f'\nTotal solve elapsed time: {solve_time:.4f} seconds')